# Load description for each variable in each pair

Recreates and extends analysis from https://github.com/amit-sharma/chatgpt-causality-pairs
Focuses on analysis of the Tübingen dataset from https://webdav.tuebingen.mpg.de/cause-effect/

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
#pip install '/content/drive/MyDrive/pywhy-llm'

In [ ]:
#pip install guidance

In [ ]:
#pip install python-dotenv

In [1]:
import sys
import os
import time


# Add the notebook setup path to sys.path
notebook_setup_path = os.path.abspath("../")
sys.path.insert(0, notebook_setup_path)

from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()



📋 Current sys.path before adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  4: 
  5: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages

🎯 Project root to add: /home/moleropa/repositories/master/TFM/pywhyllm
✅ Added local pywhyllm source to Python path: /home/moleropa/repositories/master/TFM/pywhyllm

📋 Updated sys.path after adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm ⭐
  1: /home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  4: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  5: 
  6: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages
📋 Current sys.p

In [2]:
from dotenv import load_dotenv
from typing import Dict, List, Tuple
import guidance
import os

from openai import OpenAI
from portkey_ai import createHeaders


load_dotenv()


True

In [3]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])
azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers
)


In [ ]:
# from pywhyllm.suggesters.tuebingen_model_suggester import TuebingenModelSuggester, Strategy
# modeler = TuebingenModelSuggester(llm= model)

In [ ]:
# from pywhyllm.suggesters.tuebingen_model_suggester_alba import TuebingenModelSuggester, Strategy
# modeler = TuebingenModelSuggester(llm= model)

In [4]:
from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester
modeler= SimpleModelSuggester(llm=model)

In [5]:
import pandas as pd

In [6]:
#df = pd.read_csv('/content/drive/MyDrive/pywhy-llm/pywhyllm/tuebingen_pairs.csv')
df = pd.read_csv('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/tuebingen_pairs.csv')

In [8]:
df.head()

,Unnamed: 0,var1,var2,ground_truth,truth_ab,truth_ba,context,var1_desc,var2_desc
0,pair0001,Altitude,Temperature,R,1,0,Information for pairs0001:\n\nDWD data (Deutsc...,Altitude refers to the height of an object or ...,Temperature is a measure of the average kineti...
1,pair0002,Altitude,Precipitation,R,1,0,Information for pairs0002:\n\nDWD data (Deutsc...,Altitude is a geographical concept referring t...,Precipitation is a meteorological phenomenon t...
2,pair0003,Longitude,Temperature,R,1,0,Information for pairs0003:\n\nDWD data (Deutsc...,Longitude is a geographic coordinate that spec...,Temperature is a quantitative measure of the d...
3,pair0004,Altitude,Sunshine hours,R,1,0,Information for pairs0004:\n\nDWD data (Deutsc...,Altitude is a geographical term referring to t...,Sunshine hours refer to the total number of ho...
4,pair0005,Age,Length,R,1,0,Information for pairs0005:\n\nhttps://archive....,"In the context of the Abalone dataset, the con...","In the context of the Abalone dataset, 'Length..."


# Get relationship of each variable pair

In [7]:
llm_output : Dict[str, dict] = {}

####  Variables + Straight Strategy

## ALBA TEST
modify these variables to run tests or run everything completely

#TO DO: ready to run everything

In [8]:
# Define parameters for the experiment
temperature = 0.3
num_runs = 1 # Reduced for testing

# Create saved_pairs_info to store ground truth and variable info
saved_pairs_info = {}

# Process only the first 3 rows for testing
test_df = df.head(1) #or just df all dataset
#test_df = df #or just df all dataset
    

In [9]:
# Iterate through each pair and run multiple times
start = time.time()

for pair_number, values in test_df.iterrows():
    pair_id = f"pair{pair_number:04d}"  # Create pair ID like "pair0001", "pair0002", etc.

    saved_pairs_info[pair_id] = {
        "var1": values['var1'],
        "var2": values['var2'],
        "ground_truth": values['ground_truth'],  # columna en tu dataframe
    }
    
    for n in range(1, num_runs + 1):  # Run 1 to 5
        temp_dict = {}
        
        print(f"Processing {pair_id}, run {n}/{num_runs}")
        
        # Test A -> B direction
        temp_dict['llm_ab'] = modeler.suggest_relationship(
            variable_a=values['var1'], 
            variable_b=values['var2'], 
            description_a=values['var1_desc'], 
            description_b=values['var2_desc'], 
            strategy=Strategy.Straight,
            domain = True,
            ask_reference=True,
            confidence_level=True

        )
        
        # Test B -> A direction  
        temp_dict['llm_ba'] = modeler.suggest_relationship(
            variable_a=values['var2'], 
            variable_b=values['var1'], 
            description_a=values['var2_desc'], 
            description_b=values['var1_desc'], 
            strategy=Strategy.Straight,
            domain = True,
            ask_reference=True,
            confidence_level=True
        )
        
        # Store results with key: (pair_id, temperature, run_number)
        llm_output[(pair_id, temperature, n )] = temp_dict
        
        print(f"  A->B: {temp_dict['llm_ab']}, B->A: {temp_dict['llm_ba']}")

# Calculate latencies after all processing is complete
total_time = time.time() - start
print(f"\nCompleted processing {len(test_df)} pairs with {num_runs} runs each")
print(f"Total execution time: {total_time:.2f}s")

# Calculate latencies
avg_latency_per_pair = total_time / (len(test_df) * num_runs)  # Time per pair (both A->B and B->A) per run
avg_latency_per_run = total_time / (len(test_df) * num_runs * 2)  # Time per individual query (A->B or B->A)

print(f"Average time per pair (A->B + B->A): {avg_latency_per_pair:.2f}s")
print(f"Average time per individual query: {avg_latency_per_run:.2f}s")

Processing pair0000, run 1/1


AttributeError: 'SimpleModelSuggester' object has no attribute 'suggest_relationship'

In [13]:
temp_dict

{'llm_ab': [' Altitude',
  ' Temperature',
  'To evaluate the causal relationship between altitude and temperature, we need to consider how these two factors interact with one another.\n\nAtmospheric science indicates that, generally, as altitude increases, temperature tends to decrease. This is due to the fact that air pressure decreases with altitude, leading to a lower temperature in the atmosphere above. Thus, we can say that altitude can influence temperature.\n\nOn the other hand, temperature does not cause changes in altitude. Temperature variations (such as those caused by weather conditions) do not alter the physical height of a geographic location. Therefore, it is incorrect to say that temperature causes altitude.\n\nGiven this reasoning, the most accurate statement is that altitude causes temperature to change. \n\nThus, the best option is:\n\n<answer>A</answer>'],
 'llm_ba': [None,
  None,
  'To evaluate the cause-and-effect relationship between temperature and altitude, w

In [15]:
llm_output

{('pair0000',
  0.3,
  1): {'llm_ab': (1,
   1.0,
   ['T. J. McGregor, "Effects of Altitude on Temperature Variation in Mountain Areas," 2018',
    'J. K. Halverson, "The Impact of Altitude on Climate Zones and Temperature Patterns," 2020',
    'L. R. Smith and M. T. Jones, "Altitude and Its Relationship with Temperature in Diverse Ecosystems," 2021']), 'llm_ba': (0,
   0.9,
   ['Betts, A. K., "The impact of temperature on the vertical profile of humidity in the atmosphere", 2000',
    'Stull, R. B., "Meteorology for Scientists and Engineers", 2017',
    'Barry, R. G., & Choudhury, J. R., "Climate Change and Mountain Regions: A Review", 2009'])}}

### Hallbayes

In [14]:
# Create hallbayes backend using the existing Azure OpenAI client
# We need to bypass the default OpenAI API key requirement
import os
from hallbayes import OpenAIBackend, OpenAIItem, OpenAIPlanner
temp_api_key = os.environ.get("PORTKEY_AZURE_US_API_KEY")

# Create the backend with our Azure model and then replace the client
hallbayes_backend = OpenAIBackend(model=azure_model, api_key=temp_api_key)

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


# Replace the default client with our configured Azure OpenAI client 
hallbayes_backend.client = azure_openai_client


In [15]:
planner = OpenAIPlanner(hallbayes_backend, temperature=0.3)

In [16]:
# HallBayes validation function adapted for llm_output structure
def validate_llm_output_with_hallbayes(llm_output, saved_pairs_info, planner, method="with_evidence", threshold=0.10, n_runs=1):
    """
    Validates causal relationships from llm_output using hallbayes with hallucination risk assessment
    
    Args:
        llm_output: dict with keys (pair_id, temperature, run) containing llm_ab and llm_ba results
        saved_pairs_info: dict with pair information (var1, var2, ground_truth)
        planner: configured hallbayes planner
        method: "closed_book" or "with_evidence" 
        threshold: max acceptable hallucination risk
        n_runs: number of validation runs per relationship (for consistency)
    
    Returns:
        dict: hallucination risk results for each query
    """
    print(f"Calculating hallucination risk using {method.upper()} method")
    print(f"Threshold: {threshold:.1%}")
    print("=" * 60)
    
    hallucination_risks = {}
    
    # Process each pair and run combination
    for (pair_id, temp, run_num), output in llm_output.items():
        print(f"\nProcessing {pair_id}, run {run_num}")
        
        pair_info = saved_pairs_info[pair_id]
        var1, var2 = pair_info['var1'], pair_info['var2']
        
        # Process A→B direction
        ab_result = output['llm_ab']
        ab_response = ab_result.get('result', [None, None, ""])[2]  # Get the LLM response text
        ab_evidence = ab_result.get('sources_content', "")  # Get retrieved evidence
        
        # Process B→A direction  
        ba_result = output['llm_ba']
        ba_response = ba_result.get('result', [None, None, ""])[2]  # Get the LLM response text
        ba_evidence = ba_result.get('sources_content', "")  # Get retrieved evidence
        
        # Validate A→B
        ab_risk = _calculate_single_hallucination_risk(
            var1, var2, ab_response, ab_evidence, planner, method, threshold, n_runs
        )
        
        # Validate B→A
        ba_risk = _calculate_single_hallucination_risk(
            var2, var1, ba_response, ba_evidence, planner, method, threshold, n_runs
        )
        
        # Store results
        key = (pair_id, temp, run_num)
        hallucination_risks[key] = {
            'ab_hallucination_risk': ab_risk['avg_risk'],
            'ab_is_valid': ab_risk['final_valid'],
            'ba_hallucination_risk': ba_risk['avg_risk'], 
            'ba_is_valid': ba_risk['final_valid']
        }
        
        print(f"  A→B: Risk={ab_risk['avg_risk']:.1%}, Valid={ab_risk['final_valid']}")
        print(f"  B→A: Risk={ba_risk['avg_risk']:.1%}, Valid={ba_risk['final_valid']}")
    
    return hallucination_risks


def _calculate_single_hallucination_risk(cause, effect, response, evidence, planner, method, threshold, n_runs):
    """
    Calculate hallucination risk for a single causal relationship query
    """
    # Prepare prompt based on method
    if method == "closed_book":
        prompt = f"""
        Causal Knowledge Assessment:
        
        Claim: "{cause}" causes "{effect}"
        Response: {response}
        
        Question: Based on established scientific knowledge, 
        is this causal claim and response scientifically accurate?
        
        Answer: Yes/No with brief justification.
        """
    elif method == "with_evidence":
        prompt = f"""
        Evidence-Based Assessment:
        
        Retrieved Evidence: {evidence}
        
        Claim: "{cause}" causes "{effect}"
        LLM Response: {response}
        
        Question: Based on the provided evidence and your knowledge, 
        is this causal relationship claim and response valid and well-supported?
        
        Answer: Yes/No referencing the evidence and additional knowledge.
        """
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Run multiple validations for consistency
    run_results = []
    valid_count = 0
    
    for run in range(n_runs):
        try:
            item = OpenAIItem(prompt=prompt, n_samples=3, m=4, skeleton_policy=method)
            metrics = planner.run([item], h_star=threshold, isr_threshold=1.0)
            
            if metrics:
                metric = metrics[0]
                is_valid = metric.decision_answer
                risk = metric.roh_bound
                
                if is_valid:
                    valid_count += 1
                
                run_results.append({
                    'valid': is_valid,
                    'risk': risk,
                    'run': run + 1
                })
            else:
                run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': 'No metrics'})
                
        except Exception as e:
            run_results.append({'valid': False, 'risk': 1.0, 'run': run + 1, 'error': str(e)})
    
    # Calculate final decision and average risk
    avg_risk = sum(r.get('risk', 1.0) for r in run_results) / len(run_results) if run_results else 1.0
    
    # Determine final validation (majority vote)
    final_valid = valid_count > (n_runs / 2) if n_runs > 0 else False
    
    return {
        'final_valid': final_valid,
        'avg_risk': avg_risk,
        'valid_runs': valid_count,
        'total_runs': n_runs,
        'runs': run_results
    }


# Calculate hallucination risks for all queries
print("Calculating hallucination risks for all A→B and B→A queries...")
hallucination_results = validate_llm_output_with_hallbayes(
    llm_output, saved_pairs_info, planner, 
    method="with_evidence", threshold=0.10, n_runs=1
)

Calculating hallucination risks for all A→B and B→A queries...
Calculating hallucination risk using WITH_EVIDENCE method
Threshold: 10.0%

Processing pair0000, run 1


AttributeError: 'list' object has no attribute 'get'

### Export results

In [ ]:
# my code initial
# results : Dict = {}

# for id in saved_pairs_info:

#     av_correct_ab = 0
#     av_correct_ba = 0

#     for i in range(num_runs):

#         if llm_output[(id, 0.3, i+1)]['llm_ab'] == 1 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ab += 1
#         elif llm_output[(id, 0.3, i+1)]['llm_ab'] == 0 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ab += 1

#         if llm_output[(id, 0.3, i+1)]['llm_ba'] == 1 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ba += 1
#         elif llm_output[(id, 0.3, i+1)]['llm_ba'] == 0 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ba += 1

#     av_correct_ab /= num_runs
#     av_correct_ba /= num_runs

#     temp : Dict = {}

#     temp['PairID'] = id
#     temp['CorrectACauseB'] = av_correct_ab
#     temp['CorrectBCauseA'] = av_correct_ba
#     temp['VarA'] = saved_pairs_info[id]['var1']
#     temp['VarB'] = saved_pairs_info[id]['var2']
#     temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']

#     results[id] = temp
#     print(results[id])




{'PairID': 'pair0000', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Temperature', 'GroundTruth': ' R'}
{'PairID': 'pair0001', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Precipitation', 'GroundTruth': ' R'}
{'PairID': 'pair0002', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Longitude', 'VarB': ' Temperature', 'GroundTruth': ' R'}


In [1]:
results : Dict = {}

for id in saved_pairs_info:

    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect confidence scores and references across runs
    confidence_scores_ab = []
    confidence_scores_ba = []
    references_ab = []
    references_ba = []

    for i in range(num_runs):
        # Extract relationship, confidence, and references from tuple
        ab_result = llm_output[(id, 0.3, i+1)]['llm_ab']
        ba_result = llm_output[(id, 0.3, i+1)]['llm_ba']
        
        # Unpack the tuples
        ab_relationship, ab_confidence, ab_refs = ab_result
        ba_relationship, ba_confidence, ba_refs = ba_result
        
        # Store confidence scores (only if not None)
        if ab_confidence is not None:
            confidence_scores_ab.append(ab_confidence)
        if ba_confidence is not None:
            confidence_scores_ba.append(ba_confidence)
            
        # Store references
        references_ab.extend(ab_refs)
        references_ba.extend(ba_refs)

        # Check correctness using the relationship value
        if ab_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ab += 1
        elif ab_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ab += 1

        if ba_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ba += 1
        elif ba_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ba += 1

    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    # Calculate average confidence scores
    avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
    avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None

    temp : Dict = {}

    temp['PairID'] = id
    temp['CorrectACauseB'] = av_correct_ab
    temp['CorrectBCauseA'] = av_correct_ba
    temp['VarA'] = saved_pairs_info[id]['var1']
    temp['VarB'] = saved_pairs_info[id]['var2']
    temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']
    temp['ConfidenceAB'] = avg_confidence_ab
    temp['ConfidenceBA'] = avg_confidence_ba
    temp['ReferencesAB'] = references_ab
    temp['ReferencesBA'] = references_ba

    results[id] = temp
    print(results[id])

NameError: name 'Dict' is not defined

In [ ]:
# Calculate accuracy metrics
accuracy_results = {}

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_confidence = 0
sum_ba_confidence = 0
count_ab_confidence = 0
count_ba_confidence = 0


for pair_id, result in results.items():
    # Individual accuracies per pair (these are already averages from multiple runs, can be decimal)
    correct_ab = result['CorrectACauseB']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    correct_ba = result['CorrectBCauseA']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    
    # Get confidence scores
    confidence_ab = result.get('ConfidenceAB')
    confidence_ba = result.get('ConfidenceBA')
    
    # Joint accuracy: average of both directions (more nuanced approach)
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    # Sum for overall statistics (averaging across all pairs)
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # Sum confidence scores for overall statistics
    if confidence_ab is not None:
        sum_ab_confidence += confidence_ab
        count_ab_confidence += 1
    if confidence_ba is not None:
        sum_ba_confidence += confidence_ba
        count_ba_confidence += 1
    
    # STORING INDIVIDUAL
    # Store individual pair results (only decimal values, no redundant percentages)
    accuracy_results[pair_id] = {
        'PairID': pair_id,
        'VarA': result['VarA'],
        'VarB': result['VarB'],
        'GroundTruth': result['GroundTruth'],
        'AccuracyAB': correct_ab,  # Decimal value (0.0 to 1.0)
        'AccuracyBA': correct_ba,  # Decimal value (0.0 to 1.0)
        'JointAccuracy': joint_accuracy,  # Average of both directions
        'ConfidenceAB': confidence_ab,  # Confidence score for A->B
        'ConfidenceBA': confidence_ba   # Confidence score for B->A
    }
    
    print(f"Pair {pair_id}: {result['VarA']} -> {result['VarB']}")
    print(f"  Ground Truth: {result['GroundTruth']}")
    print(f"  A→B Accuracy: {correct_ab:.3f}, Confidence: {f'{confidence_ab:.3f}' if confidence_ab is not None else 'N/A'}")
    print(f"  B→A Accuracy: {correct_ba:.3f}, Confidence: {f'{confidence_ba:.3f}' if confidence_ba is not None else 'N/A'}")
    print(f"  Joint Accuracy (avg): {joint_accuracy:.3f}")
    print()


# Overall accuracy statistics (averages across all pairs)
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs

# Overall confidence statistics
overall_ab_confidence = sum_ab_confidence / count_ab_confidence if count_ab_confidence > 0 else None
overall_ba_confidence = sum_ba_confidence / count_ba_confidence if count_ba_confidence > 0 else None

print("=== OVERALL ACCURACY STATISTICS ===")
print(f"Total pairs processed: {total_pairs}")
print(f"\nMEAN ACCURACIES (across all pairs):")
print(f"A→B Mean Accuracy: {overall_ab_accuracy:.3f}")
print(f"B→A Mean Accuracy: {overall_ba_accuracy:.3f}")
print(f"Joint Mean Accuracy: {overall_joint_accuracy:.3f}")
print(f"\nMEAN CONFIDENCE SCORES:")
print(f"A→B Mean Confidence: {f'{overall_ab_confidence:.3f}' if overall_ab_confidence is not None else 'N/A'}")
print(f"B→A Mean Confidence: {f'{overall_ba_confidence:.3f}' if overall_ba_confidence is not None else 'N/A'}")
print(f"\nLATENCY METRICS:")
print(f"Average time per pair (both directions): {avg_latency_per_pair:.2f}s")
print(f"Average time per query: {avg_latency_per_run:.2f}s")

# Summary statistics (only valuable info, no redundant percentages)
accuracy_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_confidence': overall_ab_confidence,
    'ba_mean_confidence': overall_ba_confidence,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}

Pair pair0000:  Altitude ->  Temperature
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 1.000
  B→A Accuracy: 1.000, Confidence: 0.900
  Joint Accuracy (avg): 1.000
  Strict Joint (both=1.0): 1.000

=== OVERALL ACCURACY STATISTICS ===
Total pairs processed: 1

MEAN ACCURACIES (across all pairs):
A→B Mean Accuracy: 1.000
B→A Mean Accuracy: 1.000
Joint Mean Accuracy: 1.000

MEAN CONFIDENCE SCORES:
A→B Mean Confidence: 1.000
B→A Mean Confidence: 0.900

PERFECT ACCURACY COUNTS (pairs with 1.0 accuracy):
A→B Perfect: 1/1 = 1.000
B→A Perfect: 1/1 = 1.000
Both Perfect: 1/1 = 1.000

LATENCY METRICS:
Average time per pair (both directions): 24.79s
Average time per query: 12.39s


In [ ]:
# Save accuracy results to CSV
import csv

# CSV file for detailed accuracy results (now includes confidence scores)
accuracy_csv_file = "alba_accuracy_results.csv"

# Define headers for detailed accuracy results (includes confidence)
accuracy_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA", "JointAccuracy",
    "ConfidenceAB", "ConfidenceBA"
]

# Write detailed accuracy results
with open(accuracy_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=accuracy_header)
    writer.writeheader()
    for pair_id, values in accuracy_results.items():
        writer.writerow(values)

print(f"Detailed accuracy CSV file '{accuracy_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "alba_accuracy_summary.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_summary)

print(f"Summary accuracy CSV file '{summary_csv_file}' has been created.")

# Display final summary table (clean format with confidence)
print("\n=== FINAL SUMMARY TABLE ===")
print(f"{'Metric':<25} {'Mean Accuracy':<15} {'Mean Confidence':<18}")
print("-" * 60)
ab_conf_str = f"{accuracy_summary['ab_mean_confidence']:.3f}" if accuracy_summary['ab_mean_confidence'] is not None else 'N/A'
ba_conf_str = f"{accuracy_summary['ba_mean_confidence']:.3f}" if accuracy_summary['ba_mean_confidence'] is not None else 'N/A'
print(f"{'A→B':<25} {accuracy_summary['ab_mean_accuracy']:<15.3f} {ab_conf_str:<18}")
print(f"{'B→A':<25} {accuracy_summary['ba_mean_accuracy']:<15.3f} {ba_conf_str:<18}")
print(f"{'Joint Accuracy':<25} {accuracy_summary['joint_mean_accuracy']:<15.3f} {'N/A':<18}")
print(f"{'Total Pairs':<25} {accuracy_summary['total_pairs']:<15} {'N/A':<18}")

Detailed accuracy CSV file 'alba_accuracy_results.csv' has been created.
Summary accuracy CSV file 'alba_accuracy_summary.csv' has been created.

=== FINAL SUMMARY TABLE ===
Metric                    Mean Accuracy   Mean Confidence    Perfect Count   Perfect Rate
------------------------------------------------------------------------------------------
A→B                       1.000           0.917              3               1.000       
B→A                       1.000           0.850              3               1.000       
Joint Accuracy            1.000           N/A                3               1.000       
Total Pairs               3               N/A                N/A             1.000       


save to csv test

NOTE: the correctness is a mean with all the runs 
Run 5 times

In [21]:
import csv
import copy

# CSV file name
csv_file = "alba_test.csv"

# Define the CSV file's header (column names) - updated to include all fields
header = ["PairID", "VarA", "VarB", "GroundTruth", "AccuracyACauseB", "AccuracyBCauseA", 
          "ConfidenceAB", "ConfidenceBA", "ReferencesAB", "ReferencesBA"]

# Write the data to the CSV file
with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=header)
    writer.writeheader()
    for pair_id, values in results.items():
        # Convert reference lists to strings for CSV
        row = values.copy()
        # Rename keys to match header
        row['AccuracyACauseB'] = row.pop('CorrectACauseB')
        row['AccuracyBCauseA'] = row.pop('CorrectBCauseA')
        row['ReferencesAB'] = "; ".join(row['ReferencesAB']) if row['ReferencesAB'] else ""
        row['ReferencesBA'] = "; ".join(row['ReferencesBA']) if row['ReferencesBA'] else ""
        writer.writerow(row)

print(f"CSV file '{csv_file}' has been created.")

CSV file 'alba_test.csv' has been created.


### Alba end test continue pywhyllm

In [18]:
# for pair_number, values in df.iterrows():

#         temp_dict = {}


#         temp_dict['llm_ab'] = modeler.suggest_relationship(variable_a=values['var1'], variable_b=values['var2'], description_a=values['var1_desc'], description_b=values['var2_desc'], strategy=Strategy.Straight)

#         temp_dict['llm_ba'] = modeler.suggest_relationship(variable_a=values['var2'], variable_b=values['var1'], description_a=values['var2_desc'], description_b=values['var1_desc'], strategy=Strategy.Straight)

#         llm_output[(pair_number, temperature, n)] = temp_dict

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

StitchWidget(initial_height='auto', initial_width='100%', srcdoc='<!doctype html>\n<html lang="en">\n<head>\n …

NameError: name 'temperature' is not defined

Running in all dataset (not run yet)

##### Average LLM Output

In [ ]:
# av_ab = 0
# av_ba = 0

# for i in range(5):
#     av_ab += llm_output[('pair0087', 0.3, i+1)]['llm_ab']
#     av_ba += llm_output[('pair0087', 0.3, i+1)]['llm_ba']

#     print(llm_output[('pair0087', 0.3, i+1)]['llm_ab'])
#     print(llm_output[('pair0087', 0.3, i+1)]['llm_ba'])

# av_ab = av_ab/5.0
# av_ba = av_ba/5.0

# print(av_ab)
# print(av_ba)

In [ ]:
# for id in saved_pairs_info:

#     av_correct_ab = 0
#     av_correct_ba = 0

#     for i in range(5):
#         print(llm_output[(id, 0.3, i+1)]['llm_ab'])

In [ ]:
# results : Dict = {}

# for id in saved_pairs_info:

#     av_correct_ab = 0
#     av_correct_ba = 0

#     for i in range(5):

#         if llm_output[(id, 0.3, i+1)]['llm_ab'] == 1 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ab += 1
#         elif llm_output[(id, 0.3, i+1)]['llm_ab'] == 0 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ab += 1

#         if llm_output[(id, 0.3, i+1)]['llm_ba'] == 1 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ba += 1
#         elif llm_output[(id, 0.3, i+1)]['llm_ba'] == 0 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ba += 1

#     av_correct_ab /= 5.0
#     av_correct_ba /= 5.0

#     temp : Dict = {}

#     temp['PairID'] = id
#     temp['CorrectACauseB'] = av_correct_ab
#     temp['CorrectBCauseA'] = av_correct_ba
#     temp['VarA'] = saved_pairs_info[id]['var1']
#     temp['VarB'] = saved_pairs_info[id]['var2']
#     temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']

#     results[id] = temp
#     print(results[id])




#### Save to csv file

In [ ]:
import csv
import copy

# CSV file name
csv_file = "gpt-4_results_straight_prompt_w_descriptions.csv"

# Define the CSV file's header (column names)
header = ["CorrectACauseB", "CorrectBCauseA", "PairID", "VarA", "VarB", "GroundTruth"]

# Write the data to the CSV file
with open(csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=header)
    writer.writeheader()
    for pair_id, values in results.items():
        writer.writerow(values)

print(f"CSV file '{csv_file}' has been created.")
